In [18]:
import os
from typing import TypedDict, List,Union

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage,AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv()

True

In [19]:
class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]
    

In [20]:
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 也可以换成你的中转服务支持的其它 OpenAI 模型
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai-proxy.org/v1"),
)

In [21]:
def process(state: AgentState) -> AgentState:
    """处理函数，解决输入"""
    response = llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content=response.content))
    print(f"LLM response: {response.content}")
    print("Current State", state["messages"])
    return state

In [22]:
graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)
agent = graph.compile()

In [25]:
conversation_history = []
user_input = input("Enter:")
while user_input != "exit":
    conversation_history.append(HumanMessage(content=user_input))
    #整个对话历史作为输入传给LLM，LLM会根据历史对话生成回复，并将回复添加到对话历史中
    result = agent.invoke({"messages": conversation_history})
    print(result["messages"])
    conversation_history = result["messages"]
    user_input = input("Enter:")

LLM response: Hello! How can I assist you today?
Current State [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
LLM response: 你好，Lily！很高兴见到你！有什么我可以帮你的吗？
Current State [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我是lily', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，Lily！很高兴见到你！有什么我可以帮你的吗？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
[HumanMessage(content='hi', additional_

In [26]:
with open("logging.txt", "w") as file:
    file.write("Conversation History:\n")
    for message in conversation_history:
        if(isinstance(message, HumanMessage)):
            file.write(f"User: {message.content}\n" )
        elif(isinstance(message, AIMessage)):
            file.write(f"AI: {message.content}\n")
    file.write("\nEnd of Conversation\n")

print("Conversation history has been logged to logging.txt")


Conversation history has been logged to logging.txt
